# How to use variational inference (VI) for posterior sampling

When using `NLE` or `NRE`, the trained neural network approximates the likelihood (-ratio), but one still needs a sampling method to obtain posterior samples. `sbi` supports **variational inference (VI)** as an alternative to MCMC via the `VIPosterior` class.

**When to use VI over MCMC:**
- You have many parameters (>10) and MCMC is too slow.
- You need fast, cheap posterior samples after an initial training phase.
- You want a normalised approximate posterior density (via `.log_prob()`).

**Trade-offs:** VI is faster but may be less accurate than MCMC. You can refine VI posteriors with importance sampling.

## Single-observation VI

The standard workflow trains a variational distribution $q(\theta)$ to approximate the posterior $p(\theta | x_o)$ for a specific observation $x_o$.

In [ ]:
import torch
from torch import eye, ones, zeros
from torch.distributions import MultivariateNormal

from sbi.inference import NLE, likelihood_estimator_based_potential
from sbi.inference.posteriors import VIPosterior
from sbi.simulators.linear_gaussian import linear_gaussian

### Define NLE

In [ ]:
torch.manual_seed(42)

num_dim = 2
num_simulations = 5000

prior = MultivariateNormal(zeros(num_dim), eye(num_dim))
likelihood_shift = -1.0 * ones(num_dim)
likelihood_cov = 0.25 * eye(num_dim)

theta = prior.sample((num_simulations,))
x = linear_gaussian(theta, likelihood_shift, likelihood_cov)

#  train NLE
trainer = NLE(prior=prior, show_progress_bars=False)
trainer.append_simulations(theta, x)
estimator = trainer.train()

# Build potential function (log-posterior up to a constant).
potential_fn, parameter_transform = likelihood_estimator_based_potential(
    likelihood_estimator=estimator,
    prior=prior,
    x_o=None,
)

### Train VI

In [ ]:
x_o = zeros(1, num_dim)

posterior = VIPosterior(
    potential_fn,
    prior=prior,
    q="nsf",
    vi_method="rKL",
)
posterior.set_default_x(x_o)
posterior.train(show_progress_bar=True)

### Sample Stats

In [ ]:
samples = posterior.sample((1000,))
log_probs = posterior.log_prob(samples)

print(f"Samples shape: {samples.shape}")
print(f"Log-prob shape: {log_probs.shape}")
print(f"Posterior mean: {samples.mean(0)}")

## Choosing the variational family (`q`)

The `q` argument specifies the parametric family of distributions used to approximate the posterior.

| Family | `q=` | Notes |
|--------|------|-------|
| Neural Spline Flow | `"nsf"` | Good default for most problems |
| Masked Autoregressive Flow | `"maf"` | Alternative to NSF |
| Neural Autoregressive Flow | `"naf"` | More expressive, slower |
| Unconstrained NAF | `"unaf"` | Variant of NAF |
| NICE | `"nice"` | Simple, fast |
| Sum-of-Squares Polynomial Flow | `"sospf"` | Polynomial-based |
| Gaussianisation Flow | `"gf"` | Mixture of Gaussians; **recommended for 1D** |
| Full-covariance Gaussian | `"gaussian"` | Simple, fast; **recommended for 1D** |
| Diagonal Gaussian | `"gaussian_diag"` | Fastest, least expressive |

You can also pass a custom `torch.distributions.Distribution` with `sample()` and `log_prob()` methods.

> **Tip:** For 1D parameter spaces, use `q="gaussian"` or `q="gf"`

## Choosing the VI method (`vi_method`)

The `vi_method` argument determines which divergence is minimised:

| Method | `vi_method=` | Behaviour |
|--------|--------------|-----------|
| Reverse KL | `"rKL"` | **Mode-seeking**: underestimates variance, fast |
| Forward KL | `"fKL"` | **Mass-covering**: overestimates variance, covers all modes |
| Importance Weighted | `"IW"` | Tighter bound; tune `K` (number of particles) |
| Alpha divergence | `"alpha"` | Interpolates: `alpha > 1` mode-seeking, `alpha < 1` mass-covering |

## Evaluation

After training, `VIPosterior` automatically runs a quality check (using Pareto-smoothed importance sampling diagnostics by default). You can also run it manually:

In [ ]:
posterior.evaluate(quality_control_metric="psis")

Other supported metrics are `"prop"` (proportion) and `"prop_prior"` (proportion with respect to the prior).

## Amortised VI

Instead of training $q(\theta)$ for a single $x_o$, you can train a **conditional flow** $q(\theta | x)$ that generalises across observations. This is done with `train_amortized()` and requires simulation data $(\theta, x)$.

In [ ]:
amortised_posterior = VIPosterior(
    potential_fn=potential_fn,
    prior=prior,
)

amortised_posterior.train_amortized(
    theta=theta,
    x=x,
    max_num_iters=500,
    show_progress_bar=True,
    flow_type="NSF",
    num_transforms=2,
    hidden_features=32,
)

### Sampling
Once trained, you can sample for **any** observation without retraining

In [ ]:
x_o1 = zeros(1, num_dim)
x_o2 = ones(1, num_dim)

samples_1 = amortised_posterior.sample((1000,), x=x_o1)
samples_2 = amortised_posterior.sample((1000,), x=x_o2)

print(f"Posterior mean for x_o1: {samples_1.mean(0)}")
print(f"Posterior mean for x_o2: {samples_2.mean(0)}")

You can also evaluate the log-probability and sample in batched mode:

In [ ]:
log_probs = amortised_posterior.log_prob(theta[:5], x=x_o1)
print(f"Log-probs: {log_probs}")

# batched sampling
x_batch = torch.randn(10, num_dim)
batched_samples = amortised_posterior.sample_batched((100,), x=x_batch)
print(f"Batched samples shape: {batched_samples.shape}")

## Refining VI posteriors with importance sampling

A `VIPosterior` can serve as a proposal for importance sampling, which often improves accuracy:

In [ ]:
from sbi.inference import ImportanceSamplingPosterior

refined_posterior = ImportanceSamplingPosterior(
    potential_fn=potential_fn,
    proposal=posterior,
    method="sir",
)
refined_posterior.set_default_x(x_o)
refined_samples = refined_posterior.sample((1000,), oversampling_factor=32)

print(f"Refined posterior mean: {refined_samples.mean(0)}")